compare the model

In [ ]:
import gurobipy as gp
from gurobipy import GRB
from matpowercaseframes import CaseFrames
import numpy as np
from collections import defaultdict
from numpy.linalg import inv
from scipy.stats import multivariate_normal
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
class Bus:
    def __init__(self, pd, gs, gens):
        self.pd = pd
        self.gs = gs
        self.gens = gens
class Gen:
    def __init__(self, bus, pmin, pmax, pstart, cost):
        self.bus = bus
        self.pmin = pmin
        self.pmax = pmax
        self.pstart = pstart
        self.cost = cost
class Line:
    def __init__(self, rate, frombus, tobus, one_over_reactance):
        self.rate = rate
        self.frombus = frombus
        self.tobus = tobus
        self.one_over_reactance = one_over_reactance # β beta
class NetworkReference:
    def __init__(self, ref, nbus, ngen, nline, r, bus, gen, line, originalindices, B, pi, stdomega, line_prob=0.9, bus_prob=0.9):
        self.ref = ref # Dictionary of ref data
        self.nbus = nbus
        self.ngen = ngen
        self.nline = nline
        self.r = r  # Reference bus index
        self.bus = bus # list of bus
        self.gen = gen  
        self.line = line 
        self.originalindices = originalindices 
        self.B = B  # Admittance matrix
        self.pi = pi  # π Inverse reduced admittance matrix
        self.stdomega = stdomega  # stdω List of standard deviations
        self.line_prob = line_prob 
        self.bus_prob = bus_prob  

In [ ]:
def admittancematrix(ref, bus_index):
    nbus = len(ref['bus'])
    B = np.zeros((nbus,nbus))
    for br in range(len(ref['branch'])):
        f_bus = ref['branch'].F_BUS.values[br] # f_bus-1, t_bus-1 for python 0-indexing
        t_bus = ref['branch'].T_BUS.values[br]
        B[f_bus-1, t_bus-1] += (-ref['branch'].BR_X.values[br]/(ref['branch'].BR_X.values[br]**2+ref['branch'].BR_R.values[br]**2)) # imaginary part of admittance, x/(x^2+r^2)
        B[t_bus-1, f_bus-1] += (-ref['branch'].BR_X.values[br]/(ref['branch'].BR_X.values[br]**2+ref['branch'].BR_R.values[br]**2)) 
        B[f_bus-1, f_bus-1] += (ref['branch'].BR_X.values[br]/(ref['branch'].BR_X.values[br]**2+ref['branch'].BR_R.values[br]**2))
        B[t_bus-1, t_bus-1] += (ref['branch'].BR_X.values[br]/(ref['branch'].BR_X.values[br]**2+ref['branch'].BR_R.values[br]**2))
    return B
def cost(ref,p): # p[g+1] due range(ref.ngen) used simultaneously by 0- and 1-indexing
    return gp.quicksum(ref.gen[g].cost[0]*p[g+1] + ref.gen[g].cost[1]*p[g+1] + ref.gen[g].cost[2] for g in range(ref.ngen))
def networkreference(data_file, line_prob=0.9, bus_prob=0.9, sigma_scaling=0.05):
    mpc = CaseFrames(data_file)
    ref = {attr: getattr(mpc,attr) for attr in mpc.attributes}
    def generateindices(d):
        # df = d.sort_values(by=d.columns[0]).reset_index(drop=True)
        originalindices = d.index
        nindices = len(originalindices)
        # reverseindices  = dict(zip(originalindices,df[df.columns[0]]))
        reverseindices  = dict(zip(originalindices,range(1,nindices+1)))
        return nindices, originalindices, reverseindices 
    ref['gen'] =  ref['gen'][ref['gen']['GEN_STATUS'] == 1]
    ref['gencost'] = ref['gencost'].loc[ref['gen'].index]
    ngen, genindices, gen_index = generateindices(ref['gen'])
    bus_gens = defaultdict(list) # equivalence of [gen_index[g] for g in ref[:bus_gens][busindices[i]]]; list of gens connected to a bus
    for num, bus in enumerate(ref['bus'].BUS_I):
        if not bus in gen_index.values():
            bus_gens[bus] = []
        else:
            for num2, val in enumerate(gen_index.values()):
                if bus == val:
                    bus_gens[bus].append(list(gen_index.keys())[num2])
    scale_factors = np.array([ref['baseMVA']**2, mpc.baseMVA, 1])
    gen = [Gen(
        ref['gen'].GEN_BUS.values[i],
        ref['gen'].PMIN.values[i]/ref['baseMVA'],
        ref['gen'].PMAX.values[i]/ref['baseMVA'],
        ref['gen'].PG.values[i]/ref['baseMVA'],
        ref['gencost'][['COST_2','COST_1','COST_0']].values[i]*scale_factors
        ) for i in range(ngen)]
    nbus, busindices, bus_index = generateindices(ref['bus'])
    bus = [Bus(
        ref['bus'].PD.values[i]/ref['baseMVA'],
        ref['bus'].GS.values[i]/ref['baseMVA'],
        bus_gens[bus_index[i+1]] ,
        ) for i in range(nbus)]
    nline, lineindices, line_index =generateindices(ref['branch'])
    line = [Line(
            ref['branch'].RATE_A.values[l]/ref['baseMVA'],
            ref['branch'].F_BUS.values[l],
            ref['branch'].T_BUS.values[l],
            1/ ref['branch'].BR_X.values[l]
            ) for l in range(nline)]
    originalindices = {'bus':busindices, 'gen':genindices, 'line':lineindices}
    ref['ref_buses'] = ref['bus'][ref['bus'].BUS_TYPE==3]
    r = ref['ref_buses'].index[0] # since the index of bus starting from 1
    nonref_indices = [b for b in range(nbus) if b != r]
    B = admittancematrix(ref, bus_index)
    pi = np.zeros((nbus,nbus))
    pi[np.ix_(nonref_indices,nonref_indices)] = inv(B[np.ix_(nonref_indices,nonref_indices)])
    stdomega = [sigma_scaling*(ref['bus'].PD.values[b]/ref['baseMVA']) for b in range(nbus)]
    return NetworkReference(ref,nbus,ngen,nline,r,bus,gen,line,originalindices,B,pi,stdomega,line_prob,bus_prob)

In [24]:
class SingleScenarioOPF:
    def __init__(self, model, p, omega):
        self.model = model
        self.p = p
        self.omega = omega
def singlescenarioopf(ref):
    model = gp.Model()
    p = model.addVars(range(1, ref.ngen+1), vtype=GRB.CONTINUOUS, name='p',
                      lb=[ref.gen[g].pmin for g in range(ref.ngen)],
                      ub=[ref.gen[g].pmax for g in range(ref.ngen)])
    p.start = [ref.gen[g].pstart for g in range(ref.ngen)]
    omega = model.addVars(range(1, ref.nbus+1), vtype=GRB.CONTINUOUS, name='omega')
    def busvalue(ref, i): # omega[i+1] due to j in range(ref.nbus) in theta
        return gp.quicksum(p[g] for g in range(1, ref.ngen+1)) + omega[i+1] - ref.bus[i].pd - ref.bus[i].gs
    def theta(ref, busvalue, i): # ref.pi[i-1,j] due to i = ref.line[l].frombus which start from 1
        return gp.quicksum(ref.pi[i-1,j]*busvalue(ref,j) for j in range(ref.nbus))
    def lineflow(l):
        return ref.line[l].one_over_reactance*(
        theta(ref,busvalue,ref.line[l].frombus) - theta(ref,busvalue,ref.line[l].tobus)
        )
    model.addConstrs((lineflow(l) <= ref.line[l].rate for l in range(ref.nline)), name='c')
    model.addConstrs((lineflow(l) >= -ref.line[l].rate for l in range(ref.nline)), name='c')
    model.addConstr(0 == gp.quicksum(gp.quicksum(p[g] for g in ref.bus[b].gens) +omega[b+1]-ref.bus[b].pd-ref.bus[b].gs for b in range(ref.nbus)), name='c')
    model.setObjective(cost(ref,p), GRB.MINIMIZE)
    return SingleScenarioOPF(model, p, omega)
def opfscenarios_dist(ref, m, nsamples=1000):
    nonzeroindices = [i for i in range(len(ref.stdomega)) if ref.stdomega[i] > 1e-5]
    mean = np.zeros(len(nonzeroindices))
    cov = np.diag(list(map(ref.stdomega.__getitem__, nonzeroindices)))**2
    omega = multivariate_normal.rvs(mean=mean, cov=cov, size=nsamples)
    omega_samples = np.zeros((ref.nbus, nsamples))
    omega_samples[nonzeroindices] = omega.T if omega.ndim == 2 else omega[:, np.newaxis]
    return opfscenarios(ref, m, omega_samples)

In [21]:
class OPFScenarios:
    def __init__(self,noptimal, ref, scenarios, solutions, cbases, rbases, whichbasis, whichscenario):
        self.noptimal = noptimal
        self.ref = ref
        self.scenarios = scenarios
        self.solutions = solutions
        self.cbases = cbases
        self.rbases = rbases
        self.whichbasis = whichbasis
        self.whichscenario = whichscenario

def opfscenarios(ref, m, omega_samples):
    nsamples = omega_samples.shape[1]
    status = [None] * nsamples
    soln_p = np.zeros((nsamples, ref.ngen))
    cbases = {}
    rbases = {}
    noptimal = 0
    
    for s in tqdm(range(nsamples)):
        for num in m.omega:
            m.omega[num].lb = omega_samples[num-1][s] # num-1 due to m.omega start from 1
            m.omega[num].ub = omega_samples[num-1][s]
        # m.model.setParam('OutputFlag', 0) # suppress the output
        m.model.optimize()
        status[s] = m.model.status
        if status[s] == GRB.OPTIMAL:
            soln_p[s,:] = m.p.X
            noptimal += 1
            cbasis = tuple(m.model.getAttr('Vbasis', m.model.getVars()))
            rbasis = tuple(m.model.getAttr('Cbasis', m.model.getConstrs()))
            cbases[cbasis] = cbases.get(cbasis, [])
            rbases[rbasis] = rbases.get(rbasis, [])
            cbases[cbasis].append(noptimal)
            rbases[rbasis].append(noptimal)
    assert noptimal == sum(1 for stat in status if stat == GRB.OPTIMAL), 'Mismatch in optimal scenario count'
    sample_p = soln_p[np.array(status)==GRB.OPTIMAL,:]
    sample_omega = omega_samples[:, np.array(status)==GRB.OPTIMAL]
    colbases = list(cbases.keys())
    rowbases = list(rbases.keys())
    whichcol = dict(zip(colbases, range(len(colbases))))
    whichrow = dict(zip(rowbases, range(len(rowbases))))
    whichbasis = np.zeros((noptimal, 2), dtype=int)
    for ckey in cbases.keys():
        whichbasis[cbases.get(ckey)[-1]-1,0] = whichcol[ckey]
    for rkey in rbases.keys():
        whichbasis[rbases.get(rkey)[-1]-1,1] = whichrow[rkey]
    whichscenario = {}
    for i in range(noptimal):
        basiskey = (whichbasis[i,0], whichbasis[i,1])
        whichscenario[basiskey] = whichscenario.get(basiskey, [])
        whichscenario[basiskey].append(i)
    return OPFScenarios(noptimal, ref, sample_omega, sample_p, colbases, rowbases, whichbasis, whichscenario)

def get_opf_solution(m, omega_samples):
    for num, i in enumerate(omega_samples):
        m.omega[num].lb = omega_samples[num]
        m.omega[num].ub = omega_samples[num]
    m.model.optimize()
    assert m.model.status == GRB.OPTIMAL, 'bismillah'
    return m.p.X

In [25]:
data_file='pglib-opf-17.08\\pglib_opf_case30_ieee.m'
ref = networkreference(data_file, line_prob=0.9, bus_prob=0.9, sigma_scaling=0.03)
m = singlescenarioopf(ref)
scenarios = opfscenarios_dist(ref, m, nsamples = 1)
# m.model.write('gur.lp')

Warning for adding constraints: zero or small (< 1e-13) coefficients, ignored


  0%|          | 0/1 [00:00<?, ?it/s]

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 83 rows, 36 columns and 2576 nonzeros
Model fingerprint: 0xab81f931
Coefficient statistics:
  Matrix range     [7e-05, 1e+01]
  Objective range  [3e+03, 6e+03]
  Bounds range     [1e-04, 3e+00]
  RHS range        [9e-02, 3e+00]
Presolve removed 44 rows and 34 columns
Presolve time: 0.03s

Solved in 0 iterations and 0.03 seconds (0.00 work units)
Infeasible model


100%|██████████| 1/1 [00:00<00:00, 31.36it/s]


In [26]:
m.model.display()

Minimize
  3211.2558999999997 p[1] + 6179.8693 p[2]
Subject To
c[0]: 7.276185148127542 p[1] + 7.276185148127544 p[2] + 7.276185148127543 p[3] +
7.276185148127543 p[4] + 7.276185148127542 p[5] + 7.276185148127543 p[6] +
0.9226356572931615 omega[1] + 0.3979114262734321 omega[3] + 0.27232848291394546 omega[4]
+ 0.10009967314401108 omega[5] + 0.20755651213869716 omega[6] + 0.1642299509357134
omega[7] + 0.20770217756502557 omega[8] + 0.21789903217489953 omega[9] +
0.22336863411712174 omega[10] + 0.21789903217489953 omega[11] + 0.24812593087263338
omega[12] + 0.24812593087263338 omega[13] + 0.2453670455588896 omega[14] +
0.24147739373967603 omega[15] + 0.23710152161452916 omega[16] + 0.22775422818712018
omega[17] + 0.23508220653771106 omega[18] + 0.23129017421727727 omega[19] +
0.22928572679669026 omega[20] + 0.22388371479025154 omega[21] + 0.22404944381758363
omega[22] + 0.23501086268179455 omega[23] + 0.22640956006075852 omega[24] +
0.21924994356810923 omega[25] + 0.21924994356810923 omega

C:\Users\firda\AppData\Local\Temp\ipykernel_20732\1183318356.py:1: DeprecationWarning: Model.display() is deprecated
  m.model.display()


0.17556837918956458 omega[25] + 0.17556837918956458 omega[26] + 0.1788980437530362
omega[27] + 0.18385606150623057 omega[28] + 0.17889804375303608 omega[29] +
 0.1788980437530362 omega[30] <= 1.05843
c[9]: -2.1120586108606743 p[1] + -2.112058610860659 p[2] + -2.1120586108606605 p[3] +
-2.11205861086066 p[4] + -2.112058610860659 p[5] + -2.1120586108606587 p[6] +
-0.0001994229050925056 omega[1] + -0.0007534882259638831 omega[3] +
-0.000886093411116784 omega[4] + 0.0005022712755520242 omega[5] + 0.0010414586863936037
omega[6] + 0.0008240585043832738 omega[7] + -0.9305777317426875 omega[8] +
-0.010966678922616158 omega[9] + -0.017317136311995718 omega[10] + -0.010966678922616158
omega[11] + -0.013873405140175876 omega[12] + -0.013873405140176098 omega[13] +
-0.01636053878120647 omega[14] + -0.01986705782470044 omega[15] + -0.01540689654628391
omega[16] + -0.01670710173693979 omega[17] + -0.01896654195481462 omega[18] +
-0.018432580215176575 omega[19] + -0.01815033097749419 omega[20] + -0.0

In [ ]:
# class Basis_Recourse:
#     def __init__(self, ngen, fixed, varying, basiscol, linearterms, constant):
#         self.ngen = ngen
#         self.fixed = fixed
#         self.varying = varying
#         self.basiscol = basiscol
#         self.linearterms = linearterms
#         self.constant = constant
# def BasisRecourse(ref, m, cbasis, rbasis):
#     br = Basis_Recourse(ref.ngen, {}, [], {}, np.array([]), [])
#     basic_indices = []
#     for i in range(ref.ngen):
#         if scenarios.cbases[0][i] == 0:
#             basic_indices.append(i)
#         elif scenarios.cbases[0][i] == -1: # elseif cbasis[i] == :NonbasicAtLower || cbasis[i] == :Fixed
#             br.fixed[i] = m.p.lb[i] # br.fixed[i] = m.model.colLower[i]
#         elif scenarios.cbases[0][i] == -2:
#             br.fixed[i] = m.p.ub[i] # br.fixed[i] = m.model.colUpper[i]
#         else:
#             raise ValueError("Unrecognised basis status: {} at index {}".format(scenarios.cbases[0][i], i))
#     br.varying = basic_indices
#     count = 0
#     numbasic = sum(count+1 for i in scenarios.rbases[0] if i == -1) 
#     basiscol = br.basiscol = dict(zip(basic_indices, range(numbasic)))
#     assert len(basic_indices) == numbasic, "Mismatch: len(basic_indices) != numbasic"
#     assert basic_indices == sorted(basic_indices), "basic_indices is not sorted"
#     assert len(m.model.getA().toarray()) == 2*ref.nline+1, 'bismillah'
#     basis = np.zeros((numbasic, numbasic), dtype=float)
#     omega_matrix = np.zeros((numbasic, ref.nbus), dtype=float)
#     c = 0
#     for i in range(2*ref.nline+1):
#         terms = m.model.getA().toarray()[i] 
#         if scenarios.rbases[0][i] == -1:
#             c+=1
#             rhs = m.model.RHS[i]
#             for num, coeff in enumerate(terms):
#                 if num+1 <= ref.ngen:
#                     if scenarios.cbases[0][num] == 0:
#                         basis[c-1, basiscol[num]] += coeff
#                     else:
#                         rhs -= coeff*br.fixed[num]
#                 else:
#                     omega_matrix[c-1, num-ref.ngen] -= coeff
#             br.constant.append(rhs)
#     assert c == numbasic == len(br.varying) == len(br.constant), "bismillah"
#     br.linearterms = inv(basis)*omega_matrix
#     br.constant = inv(basis)*br.constant
#     return br
# def get_opf_solution_br(br,omega):
#     basic_values = br.linearterms*omega + br.constant
#     soln = np.zeros(br.ngen, dtype=float)
#     for i in br.fixed.keys():
#         soln[i] = br.fixed[i]
#     for i in br.varying:
#         soln[i] = basic_values[0][br.basiscol[i]]
#     return soln
# def theta(ref, busvalue, i):
#     return gp.quicksum(ref.pi[i,j]*busvalue(j) for j in range(ref.nbus))
# def lineflow(ref, p, omega, l):
#     def busvalue(i):
#         result = omega[i] - ref.bus[i].pd - ref.bus[i].gs
#         if ref.bus[i].gens: # True if gens is not an empty list
#             result += sum(p[g-1] for g in ref.bus[i].gens)
#         return result
#     return ref.line[l].one_over_reactance*(theta(ref,busvalue,ref.line[l].frombus-1) - theta(ref,busvalue,ref.line[l].tobus-1))
# def nviolations(ref, p, omega, atol=1e-5):
#     return ngenerationviolations(ref, p) + ntransmissionviolations(ref, p, omega)
# def ngenerationviolations(ref, p, atol=1e-5):
#     return sum(ref.gen[i].pmin - atol > p[i] for i in range(ref.ngen))+\
#            sum(ref.gen[i].pmax + atol < p[i] for i in range(ref.ngen))
# def ntransmissionviolations(ref, p, omega, atol=1e-5):
#     if omega.ndim ==2:
#         return sum(ntransmissionviolations_vector(ref, p, omega[i,:], atol=atol) for i in range(omega.shape[0]))
#     else:
#         return sum(abs(lineflow(ref, p, omega, l).getValue())> ref.line[l].rate + atol for l in range(ref.nline)) 
# def ntransmissionviolations_vector(ref, p, omega, atol=1e-5):
#     sum(abs(lineflow(ref, p, omega, l).getValue())> ref.line[l].rate + atol for l in range(ref.nline)) 
# def nviolations(ref, p, omega, atol=1e-5):
#     return ngenerationviolations(ref, p) + ntransmissionviolations(ref, p, omega)    

In [ ]:
# class EnsembleRecourse:
#     def __init__(self, ref, baseline, recoursef):
#         self.ref = ref
#         self.baseline = baseline
#         self.recoursef = recoursef
# def get_opf_solution_ensemble(er, omega):
#     incumbent_p = get_opf_solution_br(er.baseline, omega)
#     incumbent_cost = cost(er.ref, incumbent_p)
#     feasible_p = nviolations(er.ref, incumbent_p, omega)
#     for rf in er.recoursef:
#         p = get_opf_solution_br(rf, omega)
#         if nviolations(er.ref, p, omega) == 0: # feasible solution
#             curr_cost = cost(er.ref, p)
#             if not feasible_p:
#                 incumbent_cost = curr_cost
#                 incumbent_p = p
#                 feasible_p = True
#             elif curr_cost < incumbent_cost:
#                 incumbent_cost = curr_cost
#                 incumbent_p = p
#     return incumbent_p

In [ ]:
# np.random.seed(19)
# ref = scenarios.ref
# nsamples=10000
# nonzeroindices = [i for i in range(len(ref.stdomega)) if ref.stdomega[i] > 1e-5]
# mean = np.zeros(len(nonzeroindices))
# cov = np.diag(list(map(ref.stdomega.__getitem__, nonzeroindices)))**2
# omega = multivariate_normal.rvs(mean=mean, cov=cov, size=nsamples)
# omega_samples = np.zeros((ref.nbus, nsamples))
# omega_samples[nonzeroindices] = omega.T if omega.ndim == 2 else omega[:, np.newaxis]
# omega_samples

In [ ]:
# plt.figure(figsize=(12, 8))
# # You can plot all buses or just the active ones. Here we plot all buses.
# plt.imshow(omega_samples, aspect='auto', cmap='viridis')
# plt.colorbar(label="Omega Value")
# plt.xlabel("Sample Index")
# plt.ylabel("Bus Index")
# plt.title("Heatmap of Omega Samples")
# plt.show()

# active_buses = nonzeroindices
# # Gather data for only the active buses
# data = [omega_samples[bus, :] for bus in active_buses]

# plt.figure(figsize=(10, 6))
# plt.boxplot(data, labels=active_buses)
# plt.xlabel("Bus Index (Active)")
# plt.ylabel("Omega Value")
# plt.title("Distribution of Omega Samples for Active Buses")
# plt.show()

In [ ]:
# np.random.seed(19)
# ref = scenarios.ref
# nsamples=1
# nonzeroindices = [i for i in range(len(ref.stdomega)) if ref.stdomega[i] > 1e-5]
# mean = np.zeros(len(nonzeroindices))
# cov = np.diag(list(map(ref.stdomega.__getitem__, nonzeroindices)))**2
# omega = multivariate_normal.rvs(mean=mean, cov=cov, size=nsamples)
# omega_samples = np.zeros((ref.nbus, nsamples))
# omega_samples[nonzeroindices] = omega.T if omega.ndim == 2 else omega[:, np.newaxis]

# model = gp.Model()
# p = model.addVars(range(1, ref.ngen+1), vtype=GRB.CONTINUOUS, name='p',
#                     lb=[ref.gen[g].pmin for g in range(ref.ngen)],
#                     ub=[ref.gen[g].pmax for g in range(ref.ngen)])
# p.start = [ref.gen[g].pstart for g in range(ref.ngen)]
# omega = model.addVars(ref.nbus, vtype=GRB.CONTINUOUS, name='omega')
# for num in omega:
#     omega[num].lb = float(omega_samples[num])
#     omega[num].ub = float(omega_samples[num])
# def cost(ref,p):
#     return gp.quicksum(ref.gen[g].cost[0]*p[g+1] + ref.gen[g].cost[1]*p[g+1] + ref.gen[g].cost[2] for g in range(len(ref.gen)))
# def busvalue(ref, i):
#     return gp.quicksum(p[g] for g in ref.bus[i].gens) + omega[i] - ref.bus[i].pd - ref.bus[i].gs
# def theta(ref, busvalue, i):
#     return gp.quicksum(ref.pi[i,j]*busvalue(ref,j) for j in range(ref.nbus))
# def lineflow(l):
#     return ref.line[l].one_over_reactance*(
#     theta(ref,busvalue,bus_index_inv[ref.line[l].frombus]-1) - theta(ref,busvalue,bus_index_inv[ref.line[l].tobus]-1)
#     )
# model.addConstrs((lineflow(l) <= ref.line[l].rate for l in range(ref.nline)), name='c1d_ub')
# model.addConstrs((lineflow(l) >= -ref.line[l].rate for l in range(ref.nline)), name='c1d_lb')
# model.addConstr(0 == gp.quicksum(gp.quicksum(p[g] for g in ref.bus[b].gens) +omega[b]-ref.bus[b].pd-ref.bus[b].gs for b in range(ref.nbus)),
#             name='c1b')
# model.setObjective(cost(ref,p), GRB.MINIMIZE)
# # model.optimize()

In [ ]:
# model.write('gur.lp')
# https://jump.dev/JuMP.jl/v0.18/refmodel.html?highlight=write
# writeLP(m.model,"jul.lp";genericnames=false) - write the model to filename in the LP file format. Set genericnames=false for user-defined variable names.
